# Three-threshold directional-change linkage

This notebook links the two adjacent causal building blocks for `GBPUSD.r`:

1. small 0.20% to medium 0.40%;
2. medium 0.40% to large 0.60%.

A propagated small-to-medium episode must hand off through the exact same medium DCC that initiates the medium-to-large episode. This event genealogy lets us describe how far one small-origin disturbance travels through the hierarchy.

The terminal depth is hindsight until the lineage resolves. No classifier, HMM, trading rule, or independence assumption is introduced.

In [1]:
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from dc_basics.data import resolve_data_root
from dc_basics.core import dc_live_features, parse_dc_events
from dc_basics.plotting import (
    plot_dc_linkage_flow,
    plot_three_threshold_directional_change,
)

DATA_ROOT = resolve_data_root()  # $DC_DATA_ROOT or .dc_data_root
SYMBOL = "GBPUSD.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"
SMALL_THETA = 0.002
MEDIUM_THETA = 0.004
LARGE_THETA = 0.006
CASEBOOK_BARS_BEFORE = 150
CASEBOOK_BARS_AFTER = 150

if not 0.0 < SMALL_THETA < MEDIUM_THETA < LARGE_THETA:
    raise ValueError("Thresholds must satisfy 0 < small < medium < large.")

print(f"Symbol: {SYMBOL}")
print(f"Thresholds: {SMALL_THETA:.2%}, {MEDIUM_THETA:.2%}, {LARGE_THETA:.2%}")

Symbol: GBPUSD.r
Thresholds: 0.20%, 0.40%, 0.60%


In [2]:
class Scale(StrEnum):
    SMALL = "small"
    MEDIUM = "medium"
    LARGE = "large"


from dc_basics.enums import DcMode


from dc_basics.enums import AttemptOutcome


class TerminalOutcome(StrEnum):
    STOPPED_BELOW_MEDIUM = "stopped_below_medium"
    STOPPED_BELOW_LARGE = "stopped_below_large"
    REACHED_LARGE = "reached_large"
    AMBIGUOUS_SMALL_MEDIUM = "ambiguous_small_medium"
    AMBIGUOUS_MEDIUM_LARGE = "ambiguous_medium_large"
    UNRESOLVED_BELOW_MEDIUM = "unresolved_below_medium"
    UNRESOLVED_BELOW_LARGE = "unresolved_below_large"
    UNLINKED_AFTER_MEDIUM = "unlinked_after_medium"


SCALE_ORDER = (Scale.SMALL, Scale.MEDIUM, Scale.LARGE)
SCALE_SETTINGS = {
    Scale.SMALL: {"prefix": "S", "theta": SMALL_THETA},
    Scale.MEDIUM: {"prefix": "M", "theta": MEDIUM_THETA},
    Scale.LARGE: {"prefix": "L", "theta": LARGE_THETA},
}
MODE_MAP = {1.0: DcMode.UP.value, -1.0: DcMode.DOWN.value}

## Causal and genealogical contract

A clean seed begins from broad agreement on the previous bar. A small DCC then confirms the opposite small mode.

- Depth 0: the small mode returns before medium joins.
- Depth 1: medium joins the target direction, but medium later returns before large joins.
- Depth 2: medium joins and large subsequently joins the same target direction.

The medium DCC is the linkage key: its event identifier resolves the first-stage propagation and initiates the second-stage contest. Terminal depth describes the maximum confirmed scale, not a continuous monotonic price leg. Small-scale oscillations may occur while the medium-to-large contest remains active.

## Load prices and reconstruct the three clocks

In [3]:
raw = pd.read_parquet(DATA_PATH)
if not isinstance(raw.index, pd.DatetimeIndex):
    if "time" not in raw.columns:
        raise TypeError("Expected a DatetimeIndex or a 'time' column.")
    raw = raw.set_index(pd.to_datetime(raw.pop("time")))
raw = raw.sort_index()

if raw.index.has_duplicates:
    raise ValueError(f"Found {raw.index.duplicated().sum():,} duplicate timestamps.")
if "close" not in raw.columns:
    raise KeyError("The input data has no 'close' column.")
prices = raw["close"].astype(np.float64).rename(SYMBOL)
if not np.isfinite(prices.to_numpy()).all() or prices.le(0.0).any():
    raise ValueError("Close prices must be finite and strictly positive.")

print(f"Bars: {len(prices):,}")
print(f"Range: {prices.index[0]} to {prices.index[-1]}")

Bars: 258,573
Range: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


In [4]:
from dc_basics.events import prepare_event_ledger


event_ledgers = {}
live_frames = {}
mode_frame = pd.DataFrame(index=prices.index)
for scale in SCALE_ORDER:
    settings = SCALE_SETTINGS[scale]
    raw_events = parse_dc_events(prices=prices, theta=settings["theta"])
    ledger = prepare_event_ledger(raw_events, settings["prefix"], scale, settings["theta"])
    live = dc_live_features(prices, theta=settings["theta"])
    event_ledgers[scale] = ledger
    live_frames[scale] = live
    mode_frame[f"{scale.value}_mode"] = live["direction"].map(MODE_MAP)
    mode_frame[f"{scale.value}_upward_dcc"] = live["upward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_downward_dcc"] = live["downward_dcc"].astype(bool)
    mode_frame[f"{scale.value}_event_id"] = mode_frame.index.to_series().map(
        ledger.set_index("dcc_idx")["event_id"]
    )

clock_summary = pd.DataFrame.from_records([
    {
        "scale": scale.value,
        "threshold": SCALE_SETTINGS[scale]["theta"],
        "events": len(event_ledgers[scale]),
        "first_mode_time": live_frames[scale]["direction"].first_valid_index(),
    }
    for scale in SCALE_ORDER
]).set_index("scale")
clock_summary.style.format({"threshold": "{:.2%}", "events": "{:,}"})

,threshold,events,first_mode_time
scale,,,
small,0.20%,"2,697",2023-05-22 16:15:00+02:00
medium,0.40%,839,2023-05-22 17:50:00+02:00
large,0.60%,392,2023-05-23 11:30:00+02:00


## Reconstruct the two adjacent causal building blocks

Each table is independently valid as a two-threshold contest. A propagated row resolves on the upper DCC; a rejected row resolves on the lower DCC returning toward the unchanged upper mode. Same-bar propagation is contemporaneously known and is not a future label.

In [5]:
from dc_basics.hierarchy import build_adjacent_attempts


In [6]:
small_medium_attempts = build_adjacent_attempts(mode_frame, Scale.SMALL, Scale.MEDIUM)
medium_large_attempts = build_adjacent_attempts(mode_frame, Scale.MEDIUM, Scale.LARGE)
adjacent_attempts = pd.concat([small_medium_attempts, medium_large_attempts], ignore_index=True)
adjacent_attempts["outcome_information_time"] = adjacent_attempts["resolution_time"]
adjacent_attempts["outcome_is_hindsight_at_start"] = (
    adjacent_attempts["outcome_information_time"].gt(adjacent_attempts["start_time"])
)
pair_outcome_audit = adjacent_attempts.groupby(["edge", "outcome"], sort=False).agg(
    attempts=("initiating_event_id", "size"),
    median_resolution_bars=("resolution_bars", "median"),
)
pair_outcome_audit.style.format({"attempts": "{:,}", "median_resolution_bars": "{:.1f}"})

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


## Link the building blocks through the medium event ID

We retain small-to-medium attempts whose previous bar was broad agreement. A propagated first-stage row must match exactly one medium-to-large row on `small_medium.resolution_event_id == medium_large.initiating_event_id`. Direction and timestamp must also agree. Missing or ambiguous hand-offs are audited rather than forced into a depth label.

In [7]:
previous_modes = mode_frame[["small_mode", "medium_mode", "large_mode"]].shift(1)
seed_attempts = small_medium_attempts.copy()
seed_times = pd.DatetimeIndex(seed_attempts["start_time"])
for scale in SCALE_ORDER:
    seed_attempts[f"prior_{scale.value}_mode"] = previous_modes.loc[
        seed_times, f"{scale.value}_mode"
    ].to_numpy()
seed_attempts["broad_agreement_before_seed"] = (
    seed_attempts["prior_small_mode"].eq(seed_attempts["prior_medium_mode"])
    & seed_attempts["prior_medium_mode"].eq(seed_attempts["prior_large_mode"])
)
broad_seeds = seed_attempts.loc[seed_attempts["broad_agreement_before_seed"]].copy()

print(f"All small-to-medium attempts: {len(seed_attempts):,}")
print(f"Clean broad-agreement origins: {len(broad_seeds):,}")

All small-to-medium attempts: 1,768
Clean broad-agreement origins: 1,279


In [8]:
from dc_basics.lineage import build_three_scale_linkages


linkages = build_three_scale_linkages(broad_seeds, medium_large_attempts)

## Linkage integrity and exclusions

Resolved rows (the lineage reached large, or a rejection stopped it) receive depths 0, 1, or 2. A depth is the maximum confirmed scale, not a monotone price leg. Ambiguous, unresolved, or unlinked lineages have no terminal depth: they stay visible in the audit and are excluded from every probability ladder.

Strict-clean rows additionally exclude any zero-length (same-bar) stage, where the seed and the upper DCC that joins it fire on one bar. Such a stage can only be a propagation, because a rejection needs a *later* lower DCC. Removing same-bar stages therefore removes propagations only, and strict-clean reach rates are biased low. Use the all-resolved cohort for transition probabilities, and the strict-clean cohort where every stage must contain at least one live, unresolved bar (live-panel work).

In [9]:
terminal_outcome_audit = linkages.groupby("terminal_outcome", dropna=False, sort=False).agg(
    lineages=("seed_small_event_id", "size"),
    median_total_resolution_bars=("total_resolution_bars", "median"),
)
resolved_linkages = linkages.loc[linkages["terminal_depth"].notna()].copy()
strict_clean_linkages = resolved_linkages.loc[
    resolved_linkages["terminal_label_hindsight_at_seed"]
    & ~resolved_linkages["small_medium_same_bar"]
    & ~resolved_linkages["medium_large_same_bar"]
].copy()
if strict_clean_linkages.empty:
    raise ValueError("No strict-clean resolved lineages were found.")
if not strict_clean_linkages["terminal_label_hindsight_at_seed"].all():
    raise AssertionError("A clean terminal label was known at the seed.")
if not resolved_linkages.index.equals(linkages.index[linkages["resolved_lineage"]]):
    raise AssertionError("The local resolved cohort disagrees with the library's resolved_lineage flag.")
if not strict_clean_linkages.index.equals(linkages.index[linkages["strict_clean"]]):
    raise AssertionError("The local strict-clean cohort disagrees with the library's strict_clean flag.")

linkage_exclusion_audit = pd.Series({
    "broad_agreement_seeds": len(linkages),
    "resolved_lineages": len(resolved_linkages),
    "small_medium_same_bar": int(resolved_linkages["small_medium_same_bar"].sum()),
    "medium_large_same_bar": int(resolved_linkages["medium_large_same_bar"].sum()),
    "strict_clean_lineages": len(strict_clean_linkages),
})
display(terminal_outcome_audit.style.format({
    "lineages": "{:,}",
    "median_total_resolution_bars": "{:.1f}",
}))
display(linkage_exclusion_audit)

,lineages,median_total_resolution_bars
terminal_outcome,,
reached_large,392,108.0
stopped_below_medium,663,40.0
stopped_below_large,223,191.0
unresolved_below_large,1,nan


broad_agreement_seeds         1279
resolved_monotone_lineages    1278
small_medium_same_bar           36
medium_large_same_bar           15
strict_clean_lineages         1235
dtype: int64

In [10]:
linked_rows = linkages.loc[linkages["medium_large_outcome"].notna()].copy()
linked_second_stage = linked_rows.merge(
    medium_large_attempts[["initiating_event_id", "start_time", "target_mode", "prior_upper_mode"]],
    left_on="medium_event_id",
    right_on="initiating_event_id",
    how="left",
    validate="one_to_one",
    suffixes=("_lineage", "_stage_two"),
)
if linked_second_stage["initiating_event_id"].isna().any():
    raise AssertionError("A linked lineage lacks its second-stage attempt.")
if not linked_second_stage["medium_time"].eq(linked_second_stage["start_time"]).all():
    raise AssertionError("A second-stage attempt did not start at the linking medium DCC.")
if not linked_second_stage["target_mode_lineage"].eq(linked_second_stage["target_mode_stage_two"]).all():
    raise AssertionError("Target direction changed at the linkage hand-off.")

handoff_audit = pd.Series({
    "small_medium_propagations_from_broad_seeds": int(
        broad_seeds["outcome"].eq(AttemptOutcome.PROPAGATED.value).sum()
    ),
    "exact_medium_event_handoffs": len(linked_rows),
    "unlinked_after_medium": int(
        linkages["terminal_outcome"].eq(TerminalOutcome.UNLINKED_AFTER_MEDIUM.value).sum()
    ),
})
handoff_audit

small_medium_propagations_from_broad_seeds    616
exact_medium_event_handoffs                   616
unlinked_after_medium                           0
dtype: int64

## Linked transition probabilities

The following frequencies are full-sample hindsight descriptions. They are not calibrated forecasts. The multiplication identity uses one internally consistent linked cohort; it does not multiply the two unconditional adjacent-edge rates.

`P(reach large) = P(reach medium) × P(reach large | medium)` holds by arithmetic for any cohort in which reaching large implies reaching medium, so it is not evidence of anything. The check that *can* fail is genealogical: every lineage counted as reaching medium must have propagated through an exact medium hand-off, and every lineage counted as reaching large must have propagated through both edges.

Two cohorts are shown. The all-resolved cohort is the unbiased one. The strict-clean cohort drops same-bar propagations only (see above), so its reach rates are lower by construction.

In [11]:
def summarize_linked_probabilities(sample: pd.DataFrame, sample_name: str) -> dict[str, object]:
    reached_medium = sample["terminal_depth"].ge(1)
    reached_large = sample["terminal_depth"].eq(2)
    # Genealogical consistency: this can fail if the linkage were broken.
    if not sample.loc[reached_medium, "small_medium_outcome"].eq(AttemptOutcome.PROPAGATED.value).all():
        raise AssertionError("A lineage counted as reaching medium did not propagate small->medium.")
    if not sample.loc[reached_medium, "medium_event_id"].notna().all():
        raise AssertionError("A lineage counted as reaching medium has no exact medium hand-off.")
    if not sample.loc[reached_large, "medium_large_outcome"].eq(AttemptOutcome.PROPAGATED.value).all():
        raise AssertionError("A lineage counted as reaching large did not propagate medium->large.")
    if sample.loc[~reached_medium, "medium_event_id"].notna().any():
        raise AssertionError("A depth-0 lineage carries a medium hand-off.")
    return {
        "sample": sample_name,
        "lineages": len(sample),
        "p_reach_medium_from_seed": reached_medium.mean(),
        "p_reach_large_given_medium": reached_large.sum() / reached_medium.sum(),
        "p_reach_large_from_seed": reached_large.mean(),
    }


linked_probability_summary = pd.DataFrame.from_records([
    summarize_linked_probabilities(resolved_linkages, "all resolved (unbiased)"),
    summarize_linked_probabilities(strict_clean_linkages, "strict clean (no same-bar stage; biased low)"),
]).set_index("sample")
linked_probability_summary.style.format({
    "lineages": "{:,}",
    "p_reach_medium_from_seed": "{:.2%}",
    "p_reach_large_given_medium": "{:.2%}",
    "p_reach_large_from_seed": "{:.2%}",
})

,lineages,p_reach_medium_from_seed,p_reach_large_given_medium,p_reach_large_from_seed,product_check
sample,,,,,
all resolved monotone,"1,278",48.12%,63.74%,30.67%,30.67%
strict clean future,"1,235",46.32%,62.24%,28.83%,28.83%


In [12]:
from dc_basics.audit import summarize_stage


linked_medium_event_ids = set(linkages.loc[linkages["medium_event_id"].notna(), "medium_event_id"])
linked_stage_two_attempts = medium_large_attempts.loc[
    medium_large_attempts["initiating_event_id"].isin(linked_medium_event_ids)
]
stage_property_summary = pd.DataFrame.from_records([
    summarize_stage(small_medium_attempts, "all small->medium attempts"),
    summarize_stage(broad_seeds, "broad-origin small->medium"),
    summarize_stage(medium_large_attempts, "all medium->large attempts"),
    summarize_stage(linked_stage_two_attempts, "linked medium->large hand-offs"),
]).set_index("stage")
all_stage_two = stage_property_summary.loc["all medium->large attempts"]
linked_stage_two = stage_property_summary.loc["linked medium->large hand-offs"]
print(
    "Linked medium->large hand-offs are "
    f"{len(linked_stage_two_attempts):,} of {len(medium_large_attempts):,} medium->large attempts; "
    "the two rows are identical: "
    f"{all_stage_two.equals(linked_stage_two)}"
)
stage_property_summary.style.format({
    "attempts": "{:,}",
    "resolved_attempts": "{:,}",
    "propagation_rate": "{:.2%}",
    "median_resolution_bars": "{:.1f}",
    "same_bar_propagations": "{:,}",
    "same_bar_rate": "{:.2%}",
    "clean_future_attempts": "{:,}",
    "clean_propagation_rate": "{:.2%}",
    "median_clean_resolution_bars": "{:.1f}",
})

,attempts,resolved_attempts,propagation_rate,median_resolution_bars,same_bar_rate
stage,,,,,
all small->medium attempts,1768,"1,768",0.474548,34.000000,0.024887
broad-origin small->medium,1279,"1,279",0.481626,35.000000,0.028147
all medium->large attempts,616,615,0.637398,69.000000,0.024390
linked medium->large hand-offs,616,615,0.637398,69.000000,0.024390


### Reading the stage table

- `propagation_rate` counts every resolved attempt, including zero-bar (same-bar) propagations. `clean_propagation_rate` uses only attempts with at least one unresolved bar (`clean_future_attempts`), so it is lower by construction. The two are different quantities, not two estimates of one rate.
- With three scales and broad-agreement roots, the linked medium->large hand-offs are essentially *all* medium->large attempts, so the "all" and "linked" rows coincide. A medium DCC against large is almost always preceded by a small DCC that left broad agreement and propagated through that medium DCC. The near-exception is the rare case where the upper clock confirms first (see the nesting audit in `3_threshold_dc`).
- The linked and unconditional rates can genuinely differ only when the root definition is narrower (for example four-scale agreement in `4_threshold_dc_linkage`), when a filter such as strict-clean is applied to the lineages, or at deeper hand-offs. That is why the lineage, not the edge table, is the unit of analysis.

In [13]:
linkage_flow = plot_dc_linkage_flow(
    strict_clean_linkages,
    title=f"{SYMBOL}: strict-clean linked propagation depth",
)
linkage_flow.show()

## Resolution anatomy and nested small activity

Stage-one duration runs from the seed small DCC to either small rejection or medium confirmation. Stage-two duration runs from medium confirmation to either medium rejection or large confirmation. We also count small DCCs strictly inside the second-stage interval to show that a medium-to-large lineage can contain additional small-scale cycles.

In [14]:
small_dcc_positions = event_ledgers[Scale.SMALL]["dcc_pos"].to_numpy(dtype=np.int64)
strict_clean_linkages["small_dccs_inside_stage_two"] = pd.NA
has_stage_two = strict_clean_linkages["terminal_depth"].ge(1)
for row_index, lineage in strict_clean_linkages.loc[has_stage_two].iterrows():
    left_edge = np.searchsorted(small_dcc_positions, int(lineage["medium_position"]), side="right")
    right_edge = np.searchsorted(small_dcc_positions, int(lineage["terminal_position"]), side="left")
    strict_clean_linkages.loc[row_index, "small_dccs_inside_stage_two"] = right_edge - left_edge
strict_clean_linkages["small_dccs_inside_stage_two"] = (
    strict_clean_linkages["small_dccs_inside_stage_two"].astype("Int64")
)

duration_summary = strict_clean_linkages.groupby("terminal_depth", sort=True).agg(
    lineages=("seed_small_event_id", "size"),
    stage_one_bars_median=("small_medium_bars", "median"),
    stage_two_bars_median=("medium_large_bars", "median"),
    total_bars_median=("total_resolution_bars", "median"),
    total_bars_p90=("total_resolution_bars", lambda values: values.quantile(0.90)),
    small_dccs_inside_stage_two_median=("small_dccs_inside_stage_two", "median"),
    small_dccs_inside_stage_two_p90=("small_dccs_inside_stage_two", lambda values: values.quantile(0.90)),
)
duration_summary.style.format({
    "lineages": "{:,}",
    "stage_one_bars_median": "{:.1f}",
    "stage_two_bars_median": "{:.1f}",
    "total_bars_median": "{:.1f}",
    "total_bars_p90": "{:.1f}",
    "small_dccs_inside_stage_two_median": "{:.1f}",
    "small_dccs_inside_stage_two_p90": "{:.1f}",
})

D:\projects\quant-studies\signalfoundry\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,lineages,stage_one_bars_median,stage_two_bars_median,total_bars_median,total_bars_p90,small_dccs_inside_stage_two_median,small_dccs_inside_stage_two_p90
terminal_depth,,,,,,,
0,663,40.0,nan,40.0,161.6,,
1,216,31.0,131.0,197.0,463.5,1.0,3.0
2,356,32.0,53.0,121.0,321.5,0.0,2.0


## Directional and chronological stability

These tables use the hindsight terminal depth only for diagnosis. They test whether the aggregate ladder is hiding directional asymmetry or concentration in a few periods. They do not define causal thresholds or out-of-sample probabilities. They use the all-resolved cohort, the unbiased one; the strict-clean filter would remove same-bar propagations only.

In [15]:
direction_depth_summary = pd.crosstab(
    resolved_linkages["target_mode"],
    resolved_linkages["terminal_depth"],
    normalize="index",
)
direction_depth_summary.columns = [f"depth_{int(depth)}" for depth in direction_depth_summary.columns]
resolved_linkages["seed_quarter"] = (
    resolved_linkages["seed_time"].dt.tz_localize(None).dt.to_period("Q").astype(str)
)
quarter_depth_summary = resolved_linkages.groupby("seed_quarter", sort=True).agg(
    lineages=("seed_small_event_id", "size"),
    reached_medium_rate=("terminal_depth", lambda values: values.ge(1).mean()),
    reached_large_rate=("terminal_depth", lambda values: values.eq(2).mean()),
)
display(direction_depth_summary.style.format("{:.2%}"))
display(quarter_depth_summary.style.format({
    "lineages": "{:,}",
    "reached_medium_rate": "{:.2%}",
    "reached_large_rate": "{:.2%}",
}))

,depth_0,depth_1,depth_2
target_mode,,,
down,54.50%,17.68%,27.81%
up,52.85%,17.29%,29.85%


,lineages,reached_medium_rate,reached_large_rate
seed_quarter,,,
2023Q2,43,53.49%,30.23%
2023Q3,101,46.53%,28.71%
2023Q4,115,40.00%,26.09%
2024Q1,71,53.52%,39.44%
2024Q2,58,56.90%,31.03%
2024Q3,87,42.53%,26.44%
2024Q4,96,50.00%,30.21%
2025Q1,108,48.15%,31.48%
2025Q2,128,49.22%,26.56%


## Linked price-path casebook

One recent strict-clean lineage is selected for each terminal depth. Selection and the later vertical markers use hindsight and are diagnostic. Read each chart from the black seed line forward before looking at the blue medium-confirmation line or purple terminal line.

In [16]:
casebook_linkages = (
    strict_clean_linkages.sort_values("seed_time")
    .groupby("terminal_depth", sort=True, group_keys=False)
    .tail(1)
    .sort_values("terminal_depth")
)
display(casebook_linkages[[
    "seed_small_event_id", "seed_time", "target_mode", "medium_event_id",
    "medium_time", "terminal_event_id", "terminal_time", "terminal_depth",
    "small_medium_bars", "medium_large_bars", "total_resolution_bars",
]])

for lineage in casebook_linkages.itertuples(index=False):
    window_start_position = max(0, int(lineage.seed_position) - CASEBOOK_BARS_BEFORE)
    window_end_position = min(len(prices), int(lineage.terminal_position) + CASEBOOK_BARS_AFTER + 1)
    case_prices = prices.iloc[window_start_position:window_end_position]
    case_figure = plot_three_threshold_directional_change(
        prices=case_prices,
        small_events=event_ledgers[Scale.SMALL],
        medium_events=event_ledgers[Scale.MEDIUM],
        large_events=event_ledgers[Scale.LARGE],
        small_theta=SMALL_THETA,
        medium_theta=MEDIUM_THETA,
        large_theta=LARGE_THETA,
        n_prices=len(case_prices),
        title=(
            f"Depth {int(lineage.terminal_depth)} | target={lineage.target_mode} | "
            f"seed={lineage.seed_small_event_id}"
        ),
    )
    case_figure.add_vline(
        x=lineage.seed_time, line={"color": "#111827", "dash": "dot", "width": 2}
    )
    if pd.notna(lineage.medium_time):
        case_figure.add_vline(
            x=lineage.medium_time, line={"color": "#2563eb", "dash": "dot", "width": 2}
        )
    case_figure.add_vline(
        x=lineage.terminal_time, line={"color": "#7c3aed", "dash": "dot", "width": 2}
    )
    case_figure.show()

,seed_small_event_id,seed_time,target_mode,medium_event_id,medium_time,terminal_event_id,terminal_time,terminal_depth,small_medium_bars,medium_large_bars,total_resolution_bars
1277,S2695,2026-11-06 17:20:00+02:00,up,<NA>,NaT,S2696,2026-11-06 21:45:00+02:00,0,53,<NA>,53.0
1271,S2684,2026-10-30 15:45:00+02:00,down,M836,2026-10-30 17:00:00+02:00,M837,2026-11-02 10:35:00+02:00,1,15,211,226.0
1276,S2694,2026-11-05 21:15:00+02:00,down,M838,2026-11-05 23:05:00+02:00,L392,2026-11-06 08:45:00+02:00,2,22,116,138.0


# Linkage completion gate

Before introducing live linkage features or a probabilistic model, be able to explain:

1. Why the medium event ID is the only valid hand-off key between the two building blocks.
2. Why a broad-agreement origin gives the small disturbance a definite ancestry.
3. Why depth 0, 1, and 2 are terminal hindsight labels while attained depth is causal.
4. Why, with three scales and broad-agreement roots, the linked second-stage rate equals the unconditional medium-to-large rate, and which root definitions, filters or deeper ladders make the two differ.
5. Why the product identity requires one consistent linked cohort and no independence assumption, and why it holds by arithmetic rather than testing anything.
6. How small-scale cycles can occur inside an unresolved medium-to-large contest.
7. On which exact DCC each depth becomes attained and each terminal outcome becomes known.
8. Why same-bar, ambiguous, unresolved, and unlinked cases must remain explicit, and why excluding same-bar stages biases the strict-clean ladder low.
9. Why a linked lineage is a genealogy of confirmations rather than necessarily one monotonic price leg.
10. Why these transition frequencies remain descriptive structure rather than trading alpha.

The next research step is causal live development within each linked stage: what is known at the seed, what changes before medium confirmation, and what changes after the medium hand-off while large remains unresolved.